# AndinaLog 03B | WMS Orders | Diagnóstico v2

El diagnóstico conserva Bronze, añade una bandera y un motivo por campo y no modifica valores.


In [ ]:
from pathlib import Path
import sys, hashlib
import pandas as pd

ENTORNO="auto"
RUTA_PROYECTO_DRIVE="/content/drive/MyDrive/GIAD"
COLUMNAS_BRONZE=["order_id","cliente_id","producto_id","fecha_despacho","centro_distribucion","camion_id","chofer_id","cantidad_solicitada","cantidad_entregada","tiempo_entrega_prometido_hrs","tiempo_entrega_real_hrs","otif_on_time","otif_in_full","otif"]
CENTROS={"Cochabamba","La Paz","Santa Cruz","Oruro","Tarija"}
def encontrar_raiz():
    if ENTORNO=="drive" or (ENTORNO=="auto" and "google.colab" in sys.modules):
        from google.colab import drive; drive.mount('/content/drive'); return Path(RUTA_PROYECTO_DRIVE)
    for p in [Path.cwd(),*Path.cwd().parents]:
        if (p/'datasets/AndinaLog_03B_Bronce/andinalog_wms_orders.csv').is_file(): return p
    raise FileNotFoundError('No se encontró la raíz del proyecto')
RAIZ=encontrar_raiz()
RUTA_BRONZE=RAIZ/'datasets/AndinaLog_03B_Bronce/andinalog_wms_orders.csv'
SALIDAS=RAIZ/'proyecto-integrador/01_diagnostico/andinalog_wms_orders/salidas'
bronze=pd.read_csv(RUTA_BRONZE,dtype='string',encoding='utf-8-sig',keep_default_na=False)
assert list(bronze.columns)==COLUMNAS_BRONZE
df=bronze.copy();df.insert(0,'fila_bronze',range(1,len(df)+1))
for c in COLUMNAS_BRONZE: df[f'{c}_en_cuarentena']=False;df[f'{c}_motivo']=''
def marcar(c,m,motivo,cuarentena=True):
    m=pd.Series(m,index=df.index).fillna(False).astype(bool)
    if cuarentena: df.loc[m,f'{c}_en_cuarentena']=True
    previo=df.loc[m,f'{c}_motivo'];df.loc[m,f'{c}_motivo']=previo.where(previo.eq(''),previo+'; ')+motivo
print('Filas Bronze:',len(df))


## Reglas técnicas y del dominio

Las fechas sin zona declarada se interpretan como hora de Bolivia. Los resultados posteriores al despacho se distinguen de los atributos disponibles antes de despachar.


In [ ]:
for c in COLUMNAS_BRONZE: marcar(c,df[c].str.strip().eq(''),'Valor faltante',c!='cantidad_entregada')
# Claves y copias
patterns={'order_id':r'ORD-2026-\d{5}','cliente_id':r'CLI-\d{3}','producto_id':r'PROD-\d{3}','camion_id':r'CAM-\d{2}','chofer_id':r'CHO-\d{3}'}
for c,p in patterns.items(): marcar(c,df[c].str.strip().ne('') & ~df[c].str.fullmatch(p).fillna(False),f'Formato esperado: {p}')
firma=pd.util.hash_pandas_object(df[COLUMNAS_BRONZE],index=False)
k=df.order_id.str.strip(); variantes=firma.groupby(k,dropna=False).transform('nunique')
conflicto=k.ne('') & k.duplicated(False) & variantes.gt(1)
copia=df.duplicated(COLUMNAS_BRONZE,keep='first') & ~conflicto
marcar('order_id',conflicto,'Mismo order_id con atributos contradictorios')
marcar('order_id',copia,'Copia exacta posterior')
marcar('centro_distribucion',~df.centro_distribucion.str.strip().isin(CENTROS),'Centro fuera del catálogo permitido')
# Fechas: ISO o formato local DD/MM/AAAA; fecha imposible permanece inválida.
iso=pd.to_datetime(df.fecha_despacho,format='%Y-%m-%d %H:%M:%S',errors='coerce')
local=pd.to_datetime(df.fecha_despacho,format='%d/%m/%Y %H:%M',errors='coerce')
fecha=iso.fillna(local)
marcar('fecha_despacho',fecha.isna(),'Fecha imposible o formato no reconocido')
marcar('fecha_despacho',iso.isna() & local.notna(),'Formato local convertible a AAAA-MM-DD HH:MM:SS')
# Cantidades y tiempos
nums={c:pd.to_numeric(df[c].str.strip(),errors='coerce') for c in ['cantidad_solicitada','cantidad_entregada','tiempo_entrega_prometido_hrs','tiempo_entrega_real_hrs']}
for c,n in nums.items(): marcar(c,df[c].str.strip().ne('') & n.isna(),'Valor no numérico')
marcar('cantidad_solicitada',nums['cantidad_solicitada'].notna() & (nums['cantidad_solicitada'].le(0)|nums['cantidad_solicitada'].mod(1).ne(0)),'Cantidad solicitada debe ser entero positivo')
marcar('cantidad_entregada',nums['cantidad_entregada'].notna() & (nums['cantidad_entregada'].lt(0)|nums['cantidad_entregada'].mod(1).ne(0)),'Cantidad entregada debe ser entero no negativo')
marcar('cantidad_entregada',nums['cantidad_entregada'].gt(nums['cantidad_solicitada']),'Cantidad entregada supera la solicitada')
marcar('tiempo_entrega_prometido_hrs',nums['tiempo_entrega_prometido_hrs'].le(0),'Tiempo prometido debe ser positivo')
marcar('tiempo_entrega_real_hrs',nums['tiempo_entrega_real_hrs'].lt(0),'Tiempo real no puede ser negativo')
flags={c:pd.to_numeric(df[c],errors='coerce') for c in ['otif_on_time','otif_in_full','otif']}
for c,n in flags.items(): marcar(c,~n.isin([0,1]),'Bandera debe ser 0 o 1')
marcar('otif',flags['otif'].ne((flags['otif_on_time'].eq(1)&flags['otif_in_full'].eq(1)).astype(int)),'OTIF no coincide con sus componentes')
# Resultado ausente: observación y exclusión del KPI, sin declarar inválidos los atributos predespacho.
marcar('cantidad_entregada',df.cantidad_entregada.str.strip().eq(''),'Cantidad entregada ausente; no evaluable para KPI OTIF',False)


## Integridad referencial y exportación

Productos y camiones se contrastan con Silver v2. La referencia a choferes se limita al identificador Bronze y no incorpora nombres personales.


In [ ]:
rp=RAIZ/'proyecto-integrador/02_tratamiento/andinalog_productos/salidas/andinalog_productos_didactico_v2_silver.csv'
rf=RAIZ/'proyecto-integrador/02_tratamiento/andinalog_flota/salidas/andinalog_flota_didactico_v2_silver.csv'
rh=RAIZ/'datasets/AndinaLog_03B_Bronce/andinalog_hr_drivers.csv'
productos=pd.read_csv(rp,dtype='string',encoding='utf-8-sig');flota=pd.read_csv(rf,dtype='string',encoding='utf-8-sig');choferes=pd.read_csv(rh,dtype='string',encoding='utf-8-sig')
prod=df.producto_id.str.strip().str.upper(); cam=df.camion_id.str.strip().str.upper(); cho=df.chofer_id.str.strip().str.upper()
marcar('producto_id',~prod.isin(productos.producto_id_tratado),'Producto no encontrado en Productos Silver')
marcar('camion_id',~cam.isin(flota.camion_id_tratado),'Camión no encontrado en Flota Silver')
marcar('chofer_id',~cho.isin(choferes.chofer_id.str.strip().str.upper()),'Chofer no encontrado en HR Drivers Bronze')
df['en_cuarentena']=df[[f'{c}_en_cuarentena' for c in COLUMNAS_BRONZE]].any(axis=1)
publicas=['fila_bronze',*COLUMNAS_BRONZE,*[x for c in COLUMNAS_BRONZE for x in (f'{c}_en_cuarentena',f'{c}_motivo')],'en_cuarentena']
diagnosticado=df[publicas].copy();cuarentena=diagnosticado.loc[diagnosticado.en_cuarentena].copy()
metricas={'filas_bronze':len(bronze),'filas_diagnosticadas':len(diagnosticado),'filas_cuarentena':len(cuarentena),'filas_con_observaciones':int(df[[f'{c}_motivo' for c in COLUMNAS_BRONZE]].ne('').any(axis=1).sum())}
for c in COLUMNAS_BRONZE:metricas[f'cuarentena_{c}']=int(df[f'{c}_en_cuarentena'].sum())
reporte=pd.DataFrame([{'metrica':k,'valor':v} for k,v in metricas.items()])
pd.testing.assert_frame_equal(diagnosticado[COLUMNAS_BRONZE],bronze);assert len(cuarentena)==int(diagnosticado.en_cuarentena.sum())
SALIDAS.mkdir(parents=True,exist_ok=True);base='andinalog_wms_orders_didactico_v2_'
diagnosticado.to_csv(SALIDAS/(base+'diagnosticado.csv'),index=False,encoding='utf-8-sig');cuarentena.to_csv(SALIDAS/(base+'cuarentena.csv'),index=False,encoding='utf-8-sig');reporte.to_csv(SALIDAS/(base+'reporte_calidad.csv'),index=False,encoding='utf-8-sig')
print(metricas);display(diagnosticado.tail())
